<a href="https://colab.research.google.com/github/Siddesh-2004/FindYourPhone/blob/main/notebook/featureBasedLaptopRecom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [114]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Imports
# ═══════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

In [115]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Load & Clean
# ═══════════════════════════════════════════════════════════════
df = pd.read_csv("laptops.csv")

# Drop duplicates
df.drop_duplicates(inplace=True)

# Drop low-signal / non-feature columns
dead_cols = ["Unnamed: 0", "img_link", "no_of_ratings", "no_of_reviews"]
df.drop(columns=dead_cols, inplace=True)

# ── Extract numeric fields ──────────────────────────────────────

# RAM in GB
df["ram_gb"] = df["ram"].str.extract(r"(\d+)").astype(float).astype("Int64")

# Storage: extract SSD and HDD separately from combined strings like '1 TB HDD|256 GB SSD'
def extract_storage_gb(storage_str, storage_type):
    """Extract GB value for a given storage type (SSD or HDD)."""
    import re
    # Normalize
    s = str(storage_str).upper()
    # Find all segments matching the type
    pattern = r"(\d+(?:\.\d+)?)\s*(TB|GB)\s*" + storage_type
    matches = re.findall(pattern, s)
    total = 0
    for val, unit in matches:
        total += float(val) * 1024 if unit == "TB" else float(val)
    return total if total > 0 else np.nan

df["ssd_gb"] = df["storage"].apply(lambda x: extract_storage_gb(x, "SSD")).astype("Int64")
df["hdd_gb"] = df["storage"].apply(lambda x: extract_storage_gb(x, "HDD")).astype("Int64")

# Fill NaN storage with 0
df["ssd_gb"] = df["ssd_gb"].fillna(0).astype("Int64")
df["hdd_gb"] = df["hdd_gb"].fillna(0).astype("Int64")

# Display size (already numeric)
df["display_inch"] = df["display(in inch)"].astype(float)

# Rating (already numeric, fill NaN with median)
df["rating_num"] = df["rating"].fillna(df["rating"].median()).astype(float)

# Price
df["Price"] = df["price(in Rs.)"].astype(float)

# ── Processor brand & generation ───────────────────────────────
def get_processor_brand(proc):
    p = str(proc).lower()
    if "apple" in p or "m1" in p or "m2" in p:
        return "Apple"
    elif "amd" in p:
        return "AMD"
    elif "intel" in p:
        return "Intel"
    else:
        return "Other"

df["processor_brand"] = df["processor"].apply(get_processor_brand)

def get_processor_gen(proc):
    import re
    match = re.search(r"(\d+)th Gen", str(proc), re.IGNORECASE)
    if match:
        return int(match.group(1))
    elif "m1" in str(proc).lower():
        return 12   # treat M1 as comparable to 12th gen
    elif "m2" in str(proc).lower():
        return 13
    else:
        return 0

df["processor_gnrtn"] = df["processor"].apply(get_processor_gen).astype("Int64")

# ── OS → simplified category ────────────────────────────────────
def simplify_os(os_str):
    s = str(os_str).lower()
    if "mac" in s:
        return "Mac"
    elif "dos" in s:
        return "DOS"
    elif "chrome" in s:
        return "Chrome"
    else:
        return "Windows"

df["os_clean"] = df["os"].apply(simplify_os)

# ── Weight/form-factor category from name heuristics ───────────
# New dataset has no explicit weight column; infer from laptop name keywords
def infer_weight_cat(name):
    n = str(name).lower()
    if any(kw in n for kw in ["gaming", "tuf", "rog", "nitro", "legion", "helios", "predator"]):
        return 3   # Gaming
    elif any(kw in n for kw in ["thin", "light", "slim", "air", "ultrabook", "zenbook", "gram"]):
        return 2   # ThinNlight
    else:
        return 1   # Casual

df["weight_cat"] = df["name"].apply(infer_weight_cat).astype("Int64")

# Drop raw columns now replaced
df.drop(columns=["ram", "storage", "display(in inch)", "rating",
                  "price(in Rs.)", "os", "processor"], inplace=True)

print("After cleaning:", df.shape)
df.head(3)

After cleaning: (984, 11)


,name,ram_gb,ssd_gb,hdd_gb,display_inch,rating_num,Price,processor_brand,processor_gnrtn,os_clean,weight_cat
0,Lenovo Intel Core i5 11th Gen,16,512,0,15.6,4.5,62990.0,Intel,11,Windows,1
1,Lenovo V15 G2 Core i3 11th Gen,8,256,1024,15.6,4.4,37500.0,Intel,11,Windows,1
2,ASUS TUF Gaming F15 Core i5 10th Gen,8,512,0,15.6,4.4,49990.0,Intel,10,Windows,3


In [116]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Outlier Capping (IQR) & Low-Variance Drop
# ═══════════════════════════════════════════════════════════════
features = ['ram_gb', 'ssd_gb', 'hdd_gb', 'display_inch',
            'processor_gnrtn', 'rating_num', 'Price']

for col in features:
    df[col] = df[col].astype(float)
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    df[col] = df[col].clip(lower=q1 - 1.5 * iqr, upper=q3 + 1.5 * iqr)

print("After outlier capping:", df.shape)

After outlier capping: (984, 11)


In [117]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Normalize Features
# ═══════════════════════════════════════════════════════════════
df_model = df.dropna(subset=features).reset_index(drop=True)

scaler = MinMaxScaler()
df_model[features] = scaler.fit_transform(df_model[features])

print("After normalization:", df_model.shape)

After normalization: (984, 11)


In [118]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Rule-Based Filter
# ═══════════════════════════════════════════════════════════════
def rule_based_filter(df, budget=None, os=None, processor_brand=None,
                      weight_type=None):
    """
    Hard filters applied before cosine scoring.

    Parameters
    ----------
    budget           : float  — max normalized price
    os               : str    — 'Windows' | 'Mac' | 'DOS' | 'Chrome'
    processor_brand  : str    — 'Intel' | 'AMD' | 'Apple'
    weight_type      : str    — 'Casual' | 'ThinNlight' | 'Gaming'
    """
    filtered = df.copy()
    if budget is not None:
        filtered = filtered[filtered["Price"] <= budget]
    if os is not None:
        filtered = filtered[filtered["os_clean"] == os]
    if processor_brand is not None:
        filtered = filtered[filtered["processor_brand"] == processor_brand]
    if weight_type is not None:
        wmap = {"Casual": 1, "ThinNlight": 2, "Gaming": 3}
        filtered = filtered[filtered["weight_cat"] == wmap.get(weight_type, 1)]
    return filtered.reset_index(drop=True)

In [119]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Cosine Similarity Scorer and Euclidean Distance
# ═══════════════════════════════════════════════════════════════
from sklearn.neighbors import NearestNeighbors

def map_weights(feature_importance: dict) -> np.ndarray:
    """
    Converts user slider values (1–5) to weight multipliers (1.0–3.0).
    Formula: weight = 1.0 + (slider - 1) * 0.5
    Slider 1 → 1.0, Slider 3 → 2.0, Slider 5 → 3.0
    """
    return np.array([
        1.0 + (feature_importance.get(f, 3) - 1) * 0.5
        for f in features
    ])


def build_user_vector(ram_gb=8, ssd_gb=512, hdd_gb=0, display_inch=15.6,
                      processor_gnrtn=11, rating_num=4.0, Price=50000):
    raw = pd.DataFrame(
        [[ram_gb, ssd_gb, hdd_gb, display_inch, processor_gnrtn, rating_num, Price]],
        columns=features
    )
    return scaler.transform(raw)


def euclidean_similarity(user_vector, laptop_matrix, weights):
    """
    Computes normalized Euclidean similarity between user vector and all laptops.

    Steps:
      1. Apply weights to both sides
      2. Compute row-wise Euclidean distance
      3. Normalize by theoretical max distance in weighted [0,1] space
         → max possible distance = sqrt(sum of weights²)
      4. Flip to similarity: 1 - normalized_distance

    Returns array of shape (n,) with values in [0, 1].
    """
    weighted_user   = user_vector * weights           # shape (1, 7)
    weighted_matrix = laptop_matrix * weights         # shape (n, 7)

    # Row-wise Euclidean distance: shape (n,)
    diff      = weighted_matrix - weighted_user       # broadcasting
    distances = np.linalg.norm(diff, axis=1)

    # Theoretical max: farthest two points in weighted [0,1]^7 space
    max_distance = np.sqrt(np.sum(weights ** 2))

    # Normalize and flip to similarity
    return np.clip(1 - (distances / max_distance), 0, 1)


def score_laptops(candidate_df, user_vector, weights):
    """
    Hybrid score = 0.6 × cosine_similarity + 0.4 × euclidean_similarity
    Both terms are in [0, 1], higher = better match.
    """
    weighted_user   = user_vector * weights
    weighted_matrix = candidate_df[features].values * weights

    cosine_scores    = cosine_similarity(weighted_user, weighted_matrix)[0]
    euclidean_scores = euclidean_similarity(user_vector, candidate_df[features].values, weights)

    hybrid_scores = 0.6 * cosine_scores + 0.4 * euclidean_scores

    result = candidate_df.copy()
    result["match_score"] = (hybrid_scores * 100).round(2)
    return result.sort_values("match_score", ascending=False).reset_index(drop=True)


def score_laptops_knn(candidate_df, user_vector, weights, top_n=10):
    """
    KNN-based scorer. Replaces the weighted dot product with nearest-neighbor
    retrieval in the weighted feature space.
    Returns a df with match_score column (0–100), same shape as score_laptops output.
    """
    feature_cols = features  # reuse the global list from Cell 3

    X = candidate_df[feature_cols].values.astype(float)
    weight_vec = weights  # already a np.ndarray from map_weights()

    # Scale both the laptop matrix and user vector by weights
    X_weighted = X * weight_vec
    user_vec   = (user_vector * weight_vec).reshape(1, -1)

    k     = min(top_n, len(X_weighted))
    index = NearestNeighbors(n_neighbors=k, metric="euclidean")
    index.fit(X_weighted)

    distances, indices = index.kneighbors(user_vec, n_neighbors=k)
    dist = distances[0]

    # Absolute normalization — score is comparable across queries
    max_possible = np.sqrt(np.sum(weight_vec ** 2))
    scores = np.clip(100 * (1 - dist / max_possible), 0, 100).round(2)

    result = candidate_df.iloc[indices[0]].copy()
    result["match_score"] = scores
    return result.sort_values("match_score", ascending=False).reset_index(drop=True)

In [120]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — recommend() function
# ═══════════════════════════════════════════════════════════════
def recommend(budget=None, os=None, processor_brand=None,
              weight_type=None,
              ram_gb=8, ssd_gb=512, hdd_gb=0, display_inch=15.6,
              processor_gnrtn=11, rating_num=4.0, Price=50000,
              top_n=10, feature_importance=None):
    """
    End-to-end laptop recommender.

    1. Hard-filter by budget / OS / processor brand / weight category.
    2. Rank remaining laptops by hybrid similarity to user preference vector.
    3. Return top_n results.
    """
    weights = map_weights(feature_importance) if feature_importance else map_weights({})

    candidates = rule_based_filter(
        df_model,
        budget=budget, os=os,
        processor_brand=processor_brand,
        weight_type=weight_type
    )

    if candidates.empty:
        print("No laptops match your filters. Try relaxing some constraints.")
        return pd.DataFrame()

    user_vector = build_user_vector(
        ram_gb=ram_gb, ssd_gb=ssd_gb, hdd_gb=hdd_gb,
        display_inch=display_inch,
        processor_gnrtn=processor_gnrtn,
        rating_num=rating_num,
        Price=Price
    )

    ranked = score_laptops(candidates, user_vector, weights=weights)

    display_cols = ["name", "processor_brand", "os_clean", "Price", "match_score"]
    display_cols = [c for c in display_cols if c in ranked.columns]
    return ranked[display_cols].head(top_n)

In [121]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — 3 Persona Demos
# ═══════════════════════════════════════════════════════════════
personas = [
    {
        "name": "Student",
        "params": dict(
            budget=None, os="Windows", processor_brand="Intel",
            weight_type="ThinNlight",
            ram_gb=8, ssd_gb=256, hdd_gb=0, display_inch=14.0,
            processor_gnrtn=10, rating_num=3.5, Price=35000
        )
    },
    {
        "name": "Content Creator",
        "params": dict(
            budget=None, os="Windows", processor_brand="Intel",
            weight_type=None,
            ram_gb=16, ssd_gb=512, hdd_gb=0, display_inch=15.6,
            processor_gnrtn=12, rating_num=4.5, Price=85000
        )
    },
    {
        "name": "Gamer",
        "params": dict(
            budget=None, os="Windows", processor_brand="Intel",
            weight_type="Gaming",
            ram_gb=16, ssd_gb=512, hdd_gb=1024, display_inch=15.6,
            processor_gnrtn=12, rating_num=4.5, Price=100000
        )
    },
]

for p in personas:
    print(f"\n{'='*60}")
    print(f"  Persona: {p['name']}")
    print(f"{'='*60}")
    result = recommend(**p["params"])
    if not result.empty:
        print(result.to_string(index=False))


  Persona: Student
                                                   name processor_brand os_clean    Price  match_score
         Lenovo Ideapad Slim 3i (2021) Core i3 10th Gen           Intel  Windows 0.153712         0.20
                Infinix X1 Slim Series Core i7 10th Gen           Intel  Windows 0.196822         0.19
                Infinix X1 Slim Series Core i7 10th Gen           Intel  Windows 0.196822         0.19
 ASUS Zenbook 14 OLED (2022) Intel EVO Core i5 12th Gen           Intel  Windows 0.469834         0.18
                         MSI GF63 Thin Core i5 10th Gen           Intel  Windows 0.244187         0.17
                Infinix X1 Slim Series Core i5 10th Gen           Intel  Windows 0.139680         0.14
                Infinix X1 Slim Series Core i5 10th Gen           Intel  Windows 0.139680         0.14
                Infinix X1 Slim Series Core i5 10th Gen           Intel  Windows 0.139680         0.14
                               LG Gram Core i5 12th G

In [122]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — Golden Set Evaluation
# ═══════════════════════════════════════════════════════════════

def run_golden_set():
    passed = 0
    failed = 0

    def check(label, condition):
        nonlocal passed, failed
        if condition:
            print(f"  [ PASS ] {label}")
            passed += 1
        else:
            print(f"  [ FAIL ] {label}")
            failed += 1

    # ── helper: returns full rows ────────────────────────────────
    def recommend_full(budget=None, os=None, processor_brand=None,
                       weight_type=None,
                       ram_gb=8, ssd_gb=512, hdd_gb=0, display_inch=15.6,
                       processor_gnrtn=11, rating_num=4.0, Price=50000,
                       top_n=10):
        candidates = rule_based_filter(
            df_model,
            budget=budget, os=os,
            processor_brand=processor_brand,
            weight_type=weight_type
        )
        if candidates.empty:
            return pd.DataFrame()
        user_vector = build_user_vector(
            ram_gb=ram_gb, ssd_gb=ssd_gb, hdd_gb=hdd_gb,
            display_inch=display_inch,
            processor_gnrtn=processor_gnrtn,
            rating_num=rating_num,
            Price=Price
        )
        ranked = score_laptops(candidates, user_vector, map_weights({}))
        return ranked

    # ── pre-compute normalized price thresholds ──────────────────
    price_min = scaler.data_min_[features.index("Price")]
    price_max = scaler.data_max_[features.index("Price")]
    def norm_price(inr):
        return (inr - price_min) / (price_max - price_min)

    # dataset-level averages (normalized scale)
    avg_ram = df_model["ram_gb"].mean()
    avg_ssd = df_model["ssd_gb"].mean()

    def score_floor(results, percentile=75):
        return np.percentile(results["match_score"].values, percentile)

    # ───────────────────────────────────────────────
    # TEST GROUP 1 — Student
    # ───────────────────────────────────────────────
    print("\n=== Student ===")
    student = recommend_full(
        budget=norm_price(40000), os="Windows", processor_brand="Intel",
        weight_type="ThinNlight",
        ram_gb=8, ssd_gb=256, hdd_gb=0, display_inch=14.0,
        processor_gnrtn=10, rating_num=3.5, Price=35000
    )

    check(
        "Student — returned at least 1 result",
        not student.empty
    )
    check(
        "Student — all top 5 run Windows",
        (student.head(5)["os_clean"] == "Windows").all()
    )
    check(
        "Student — all top 5 are Intel",
        (student.head(5)["processor_brand"] == "Intel").all()
    )
    check(
        "Student — all top 5 are ThinNlight (weight_cat == 2)",
        (student.head(5)["weight_cat"] == 2).all()
    )
    check(
        "Student — top 5 normalized Price ≤ norm(₹40,000)",
        (student.head(5)["Price"] <= norm_price(40000) + 1e-6).all()
    )
    check(
        "Student — top 1 match score is above the 75th percentile of candidates",
        student.iloc[0]["match_score"] >= score_floor(student, 75)
    )

    # ───────────────────────────────────────────────
    # TEST GROUP 2 — Content Creator
    # ───────────────────────────────────────────────
    print("\n=== Content Creator ===")
    creator = recommend_full(
        budget=norm_price(100000), os="Windows", processor_brand="Intel",
        weight_type=None,
        ram_gb=16, ssd_gb=512, hdd_gb=0, display_inch=15.6,
        processor_gnrtn=12, rating_num=4.5, Price=85000
    )

    check(
        "Creator — returned at least 1 result",
        not creator.empty
    )
    check(
        "Creator — top 5 normalized Price ≤ norm(₹1,00,000)",
        (creator.head(5)["Price"] <= norm_price(100000) + 1e-6).all()
    )
    check(
        "Creator — top 5 avg RAM above dataset average",
        creator.head(5)["ram_gb"].mean() > avg_ram
    )
    check(
        "Creator — top 5 avg SSD above dataset average",
        creator.head(5)["ssd_gb"].mean() > creator["ssd_gb"].mean()
    )
    check(
        "Creator — top 1 match score is above the 75th percentile of candidates",
        creator.iloc[0]["match_score"] >= score_floor(creator, 75)
    )

    # ───────────────────────────────────────────────
    # TEST GROUP 3 — Gamer
    # ───────────────────────────────────────────────
    print("\n=== Gamer ===")
    gamer = recommend_full(
        budget=norm_price(120000), os="Windows", processor_brand="Intel",
        weight_type="Gaming",
        ram_gb=16, ssd_gb=512, hdd_gb=1024, display_inch=15.6,
        processor_gnrtn=12, rating_num=4.5, Price=100000
    )

    check(
        "Gamer — returned at least 1 result",
        not gamer.empty
    )
    check(
        "Gamer — all top 5 are Gaming (weight_cat == 3)",
        (gamer.head(5)["weight_cat"] == 3).all()
    )
    check(
        "Gamer — top 5 normalized Price ≤ norm(₹1,20,000)",
        (gamer.head(5)["Price"] <= norm_price(120000) + 1e-6).all()
    )
    check(
        "Gamer — top 5 avg RAM above dataset average",
        gamer.head(5)["ram_gb"].mean() > gamer["ram_gb"].mean()
    )
    check(
        "Gamer — top 1 is ranked higher than top 5 median score",
        gamer.iloc[0]["match_score"] >= gamer.head(5)["match_score"].median()
    )

    # ───────────────────────────────────────────────
    # TEST GROUP 4 — Edge Cases
    # ───────────────────────────────────────────────
    print("\n=== Edge Cases ===")

    impossible = recommend_full(budget=norm_price(100), os="Windows")
    check(
        "Impossible budget (₹100) returns empty results",
        impossible.empty
    )

    impossible_combo = recommend_full(
        budget=norm_price(200000), os="Mac", processor_brand="AMD"
    )
    check(
        "Conflicting filters (Mac + AMD) returns empty or handles gracefully",
        impossible_combo.empty or len(impossible_combo) >= 0
    )

    try:
        high_budget = recommend_full(
            budget=norm_price(500000), os="Windows",
            ram_gb=64, ssd_gb=2000,
            processor_gnrtn=13, rating_num=5.0, Price=400000
        )
        check(
            "Extreme specs (max everything) returns results without crash",
            not high_budget.empty
        )
    except Exception as e:
        check(f"Extreme specs returns results without crash — ERROR: {e}", False)

    # ───────────────────────────────────────────────
    # TEST GROUP 5 — Score Sanity
    # ───────────────────────────────────────────────
    print("\n=== Score Sanity ===")

    base = dict(
        budget=norm_price(120000), os="Windows", processor_brand="Intel",
        weight_type="Gaming", ram_gb=16, ssd_gb=512, hdd_gb=1024,
        processor_gnrtn=12, rating_num=4.5, Price=100000
    )

    high_disp = recommend_full(**base, display_inch=17.3, top_n=1)
    low_disp  = recommend_full(**base, display_inch=13.0, top_n=50)

    if not high_disp.empty and not low_disp.empty:
        top_name   = high_disp.iloc[0]["name"]
        high_score = high_disp.iloc[0]["match_score"]
        low_row    = low_disp[low_disp["name"] == top_name]
        low_score  = low_row.iloc[0]["match_score"] if not low_row.empty else 0
        check(
            "Top gaming laptop scores higher with display=17.3 vs 13.0",
            high_score >= low_score
        )

    check(
        "All match scores are in [0, 100]",
        gamer["match_score"].between(0, 100).all()
    )
    check(
        "Results are sorted descending by match_score",
        gamer["match_score"].is_monotonic_decreasing
    )

    # ───────────────────────────────────────────────
    # TEST GROUP 6 — Filter Correctness
    # ───────────────────────────────────────────────
    print("\n=== Filter Correctness ===")

    amd_results = recommend_full(
        budget=norm_price(80000), processor_brand="AMD", top_n=10
    )
    check(
        "processor_brand='AMD' — all top 10 are AMD",
        (amd_results["processor_brand"] == "AMD").all()
    )

    casual_results = recommend_full(
        budget=norm_price(50000), weight_type="Casual", top_n=10
    )
    check(
        "weight_type='Casual' — all top 10 are Casual (weight_cat == 1)",
        (casual_results["weight_cat"] == 1).all()
    )

    mac_results = recommend_full(
        budget=norm_price(200000), os="Mac", processor_brand="Apple", top_n=10
    )
    check(
        "os='Mac' + processor_brand='Apple' — all results are Mac OS",
        (mac_results["os_clean"] == "Mac").all() if not mac_results.empty else True
    )

    # ───────────────────────────────────────────────
    # TEST GROUP 7 — Cross-Persona Comparison
    # ───────────────────────────────────────────────
    print("\n=== Cross-Persona Comparison ===")

    student_cross = recommend_full(
        budget=norm_price(40000), os="Windows", weight_type="ThinNlight",
        ram_gb=8, ssd_gb=256,
        processor_gnrtn=10, rating_num=3.5, Price=35000, top_n=5
    )
    gamer_cross = recommend_full(
        budget=norm_price(120000), os="Windows", weight_type="Gaming",
        ram_gb=16, ssd_gb=512, hdd_gb=1024,
        processor_gnrtn=12, rating_num=4.5, Price=100000, top_n=5
    )

    check(
        "Gamer top 5 has higher avg RAM than Student top 5",
        gamer_cross["ram_gb"].mean() > student_cross["ram_gb"].mean()
    )
    check(
        "Student top 5 has lower avg Price than Gamer top 5",
        student_cross["Price"].mean() < gamer_cross["Price"].mean()
    )
    check(
        "Gamer top 5 has higher avg SSD than Student top 5",
        gamer_cross["ssd_gb"].mean() >= student_cross["ssd_gb"].mean()
    )

    # ───────────────────────────────────────────────
    # SUMMARY
    # ───────────────────────────────────────────────
    total = passed + failed
    print(f"\n{'═'*50}")
    print(f"  Results: {passed}/{total} passed", end="  ")
    print("✓ All good!" if failed == 0 else f"✗ {failed} assertion(s) failed — check above")
    print(f"{'═'*50}")

run_golden_set()


=== Student ===
  [ PASS ] Student — returned at least 1 result
  [ PASS ] Student — all top 5 run Windows
  [ PASS ] Student — all top 5 are Intel
  [ PASS ] Student — all top 5 are ThinNlight (weight_cat == 2)
  [ PASS ] Student — top 5 normalized Price ≤ norm(₹40,000)
  [ PASS ] Student — top 1 match score is above the 75th percentile of candidates

=== Content Creator ===
  [ PASS ] Creator — returned at least 1 result
  [ PASS ] Creator — top 5 normalized Price ≤ norm(₹1,00,000)
  [ PASS ] Creator — top 5 avg RAM above dataset average
  [ FAIL ] Creator — top 5 avg SSD above dataset average
  [ PASS ] Creator — top 1 match score is above the 75th percentile of candidates

=== Gamer ===
  [ PASS ] Gamer — returned at least 1 result
  [ PASS ] Gamer — all top 5 are Gaming (weight_cat == 3)
  [ PASS ] Gamer — top 5 normalized Price ≤ norm(₹1,20,000)
  [ FAIL ] Gamer — top 5 avg RAM above dataset average
  [ PASS ] Gamer — top 1 is ranked higher than top 5 median score

=== Edge Case